# Improvement 1: Basic Pitch Integration + Heuristic Pitch Splitting

## Problem

The conditioned model (`best_conditioned.pt`) expects a **2×128×T** binary piano-roll
input: one 128-note channel per guitar. During training, ground-truth per-guitar
note annotations were available. At **inference time on unseen audio**, we only have
the mixed recording — no per-guitar labels.

The original paper's RSE transcription network suffered from "reduced generalization"
on unseen data. We replace it with **Basic Pitch** (Spotify), an industry-grade
automatic music transcription tool that produces a single mixed MIDI output.

## Solution: Heuristic Pitch Splitting

Since Basic Pitch outputs **one** MIDI stream for the entire mix, we must split it
into two streams (Guitar 1 and Guitar 2). The key insight for classical guitar duets
is that one guitar typically plays the **melody** (higher register) and the other
plays **accompaniment** (lower register).

### Median-split with hysteresis

1. **Compute a running median pitch** over a sliding window (~0.5s)
2. Notes **above** the median → Guitar 1 (melody); notes **below** → Guitar 2 (bass)
3. **Hysteresis**: once a note is assigned to a guitar, it stays with that guitar
   for its entire duration, even if the median boundary shifts mid-note. This avoids
   rapid, jittery reassignment at the boundary.

The hysteresis prevents a note that sits right at the median from flipping between
Guitar 1 and Guitar 2 frame-by-frame — it gets assigned once at its onset and
remains stable.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path(".").resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import torch
import torchaudio
from src.utils.audio import load_audio
import IPython.display as ipd
from src.evaluation.bss import windowed_bss_eval

from src.models.factory import build_model
from src.models.apply import apply_model
from src.data.manifests import load_manifest
from src.utils.io import load_config

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

## 1. Transcribe with Basic Pitch

In [ ]:
try:
    from basic_pitch.inference import predict as bp_predict
    HAS_BASIC_PITCH = True
    print("basic-pitch found — using full neural AMT pipeline.")
except ImportError:
    HAS_BASIC_PITCH = False
    print(
        "basic-pitch not available (requires Python ≤3.12 due to TensorFlow dependency).\n"
        "Falling back to librosa pYIN + onset detection.\n"
        "  Install manually on Python ≤3.12:  pip install 'basic-pitch[torch]'"
    )

import librosa


def _transcribe_librosa(audio_path, sr=44100):
    """Fallback AMT using librosa pYIN pitch tracking + onset detection."""
    y, sr_loaded = librosa.load(str(audio_path), sr=sr, mono=True)
    total_samples = len(y)

    onset_samples = librosa.onset.onset_detect(
        y=y, sr=sr_loaded, units="samples", backtrack=True
    )
    if len(onset_samples) == 0:
        return []

    onset_samples = np.sort(onset_samples)
    end_samples = np.append(onset_samples[1:], total_samples)

    f0, voiced_flag, _ = librosa.pyin(
        y,
        fmin=librosa.note_to_hz("E2"),
        fmax=librosa.note_to_hz("E6"),
        sr=sr_loaded,
    )
    hop_length = 512

    note_events = []
    for onset_s, end_s in zip(onset_samples, end_samples):
        start_sec = onset_s / sr_loaded
        end_sec = end_s / sr_loaded
        if end_sec - start_sec < 0.03:
            continue

        frame_s = librosa.samples_to_frames(onset_s, hop_length=hop_length)
        frame_e = min(librosa.samples_to_frames(end_s, hop_length=hop_length), len(f0))

        region_f0 = f0[frame_s:frame_e]
        region_voiced = voiced_flag[frame_s:frame_e]
        voiced_f0 = region_f0[region_voiced & np.isfinite(region_f0)]
        if len(voiced_f0) == 0:
            continue

        midi_pitch = int(round(librosa.hz_to_midi(float(np.median(voiced_f0)))))
        midi_pitch = max(0, min(127, midi_pitch))
        note_events.append((start_sec, end_sec, midi_pitch, 0.8))

    return note_events


def transcribe_mix(audio_path, onset_threshold=0.5, frame_threshold=0.3):
    """Transcribe a mix audio file into note events.

    Uses Basic Pitch when available; falls back to librosa pYIN otherwise.
    Returns list of (start_sec, end_sec, midi_pitch, amplitude).
    """
    if HAS_BASIC_PITCH:
        _, _, note_events = bp_predict(
            str(audio_path),
            onset_threshold=onset_threshold,
            frame_threshold=frame_threshold,
        )
    else:
        note_events = _transcribe_librosa(audio_path)

    print(f"Transcribed {len(note_events)} notes from {Path(audio_path).name}")
    return note_events

## 2. Heuristic Pitch Splitting: Median-split with Hysteresis

**How it works:**

1. Build a frame-level pitch histogram: at each time frame, collect all active MIDI pitches
2. Compute a **running median pitch** over a window (default 0.5s). This adapts to local
   register changes — e.g., if both guitars shift up during a passage
3. At each note's **onset**, check if the pitch is above or below the median at that time:
   - Above → Guitar 1 (melody)
   - Below → Guitar 2 (accompaniment)
4. **Hysteresis**: the assignment is made once at onset and persists for the note's full
   duration. Even if the median shifts while the note is sounding, the note stays with
   its assigned guitar. This prevents audible switching artifacts.

In [ ]:
def median_split_with_hysteresis(
    note_events,
    audio_length_samples,
    sr=44100,
    window_sec=0.5,
    hysteresis_semitones=2,
):
    """Split mixed Basic Pitch notes into two guitar piano rolls.

    Args:
        note_events: list of (start_sec, end_sec, midi_pitch, amplitude, ...)
        audio_length_samples: total length of the audio in samples
        sr: sample rate
        window_sec: running median window in seconds
        hysteresis_semitones: half-width of the dead zone around the median.
            Notes within median ± hysteresis follow the *previous* assignment
            for that pitch, reducing rapid switching.

    Returns:
        piano_roll: torch.Tensor of shape (256, audio_length_samples)
            [0:128] = Guitar 1 notes, [128:256] = Guitar 2 notes
    """
    T = audio_length_samples

    # Step 1: Build a frame-level weighted pitch sum for running median
    pitch_sum = np.zeros(T, dtype=np.float64)
    pitch_count = np.zeros(T, dtype=np.float64)

    for ev in note_events:
        start_s, end_s, pitch = ev[0], ev[1], int(ev[2])
        s = int(start_s * sr)
        e = min(int(end_s * sr), T)
        if s < e:
            pitch_sum[s:e] += pitch
            pitch_count[s:e] += 1

    # Average pitch per frame (where notes are active)
    active = pitch_count > 0
    avg_pitch = np.zeros(T, dtype=np.float64)
    avg_pitch[active] = pitch_sum[active] / pitch_count[active]

    # Step 2: Running median via convolution (smoothed average pitch)
    win = int(window_sec * sr)
    if win < 1:
        win = 1
    kernel = np.ones(win) / win
    # Smooth the average pitch; inactive frames contribute 0 but we weight
    smoothed_pitch = np.convolve(pitch_sum, kernel, mode="same")
    smoothed_count = np.convolve(pitch_count, kernel, mode="same")
    smoothed_count = np.maximum(smoothed_count, 1e-8)
    running_median = smoothed_pitch / smoothed_count

    # Fill gaps: where no notes are active, use global median
    all_pitches = [int(ev[2]) for ev in note_events]
    global_median = float(np.median(all_pitches)) if all_pitches else 60.0
    no_activity = smoothed_pitch == 0
    running_median[no_activity] = global_median

    # Step 3: Assign each note with hysteresis
    g1_roll = np.zeros((128, T), dtype=np.float32)  # melody
    g2_roll = np.zeros((128, T), dtype=np.float32)  # accompaniment

    # Track last assignment per pitch for hysteresis
    last_assignment = {}  # pitch -> 1 or 2

    # Sort by onset time for temporal coherence
    sorted_events = sorted(note_events, key=lambda x: x[0])

    for ev in sorted_events:
        start_s, end_s, pitch = ev[0], ev[1], int(ev[2])
        s = int(start_s * sr)
        e = min(int(end_s * sr), T)
        if s >= e or pitch < 0 or pitch >= 128:
            continue

        # Median at onset
        onset_median = running_median[min(s, T - 1)]
        diff = pitch - onset_median

        if abs(diff) <= hysteresis_semitones:
            # Within dead zone: use previous assignment if available
            assignment = last_assignment.get(pitch, 1 if diff >= 0 else 2)
        else:
            assignment = 1 if diff > 0 else 2

        last_assignment[pitch] = assignment

        if assignment == 1:
            g1_roll[pitch, s:e] = 1.0
        else:
            g2_roll[pitch, s:e] = 1.0

    piano_roll = np.concatenate([g1_roll, g2_roll], axis=0)  # (256, T)
    return torch.from_numpy(piano_roll)


print("Pitch splitting function defined.")

## 3. Load the conditioned model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

config = load_config(REPO_ROOT / "configs" / "conditioned.yaml")
model_cond = build_model(config["model"]["name"], config["model"].get("kwargs", {}))

ckpt_path = REPO_ROOT / "outputs" / "checkpoints" / "best_conditioned.pt"
payload = torch.load(ckpt_path, map_location=device, weights_only=False)
state = payload.get("model_state_dict", payload)
model_cond.load_state_dict(state)
model_cond.to(device).eval()
print(f"Loaded conditioned model from {ckpt_path.name}")
print(f"note_conditioning = {model_cond.note_conditioning}")

## 4. Also load the unconditioned model (for comparison)

In [ ]:
config_u = load_config(REPO_ROOT / "configs" / "unconditioned.yaml")
model_uncond = build_model(config_u["model"]["name"], config_u["model"].get("kwargs", {}))

ckpt_u = REPO_ROOT / "outputs" / "checkpoints" / "best_unconditioned.pt"
payload_u = torch.load(ckpt_u, map_location=device, weights_only=False)
state_u = payload_u.get("model_state_dict", payload_u)
model_uncond.load_state_dict(state_u)
model_uncond.to(device).eval()
print(f"Loaded unconditioned model from {ckpt_u.name}")

## 5. Select a test track and run the full pipeline

In [ ]:
manifest = load_manifest(REPO_ROOT / config["dataset"]["manifest"], resolve_root=REPO_ROOT)
test_entries = [e for e in manifest if e["split"] == config["dataset"]["test_split"]]
print(f"Test tracks: {len(test_entries)}")
for i, e in enumerate(test_entries):
    print(f"  [{i}] {e['track_name']}")

In [ ]:
TRACK_IDX = 0  # <-- change this

entry = test_entries[TRACK_IDX]
print(f"Track: {entry['track_name']}")

mix, sr = load_audio(entry["mix"])
g1_ref, _ = load_audio(entry["sources"]["guitar1"])
g2_ref, _ = load_audio(entry["sources"]["guitar2"])
print(f"Duration: {mix.shape[-1]/sr:.2f}s")

### 5a. Transcribe the mix with Basic Pitch

In [ ]:
note_events = transcribe_mix(entry["mix"])

### 5b. Split notes with median + hysteresis

In [ ]:
piano_roll = median_split_with_hysteresis(
    note_events,
    audio_length_samples=mix.shape[-1],
    sr=sr,
    window_sec=0.5,
    hysteresis_semitones=2,
)

print(f"Piano roll shape: {piano_roll.shape}")
print(f"Guitar 1 active frames: {(piano_roll[:128].sum(0) > 0).sum().item()}")
print(f"Guitar 2 active frames: {(piano_roll[128:].sum(0) > 0).sum().item()}")

### 5c. Visualize the split piano roll

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 6), sharex=True)

# Downsample for visualization (every 1000th sample)
ds = 1000
g1_vis = piano_roll[:128, ::ds].numpy()
g2_vis = piano_roll[128:, ::ds].numpy()
time_axis = np.arange(g1_vis.shape[1]) * ds / sr

axes[0].imshow(g1_vis, aspect="auto", origin="lower",
               extent=[0, time_axis[-1], 0, 128], cmap="Blues")
axes[0].set_ylabel("MIDI pitch")
axes[0].set_title("Guitar 1 (melody — above median)")
axes[0].set_ylim(35, 100)

axes[1].imshow(g2_vis, aspect="auto", origin="lower",
               extent=[0, time_axis[-1], 0, 128], cmap="Oranges")
axes[1].set_ylabel("MIDI pitch")
axes[1].set_xlabel("Time (s)")
axes[1].set_title("Guitar 2 (accompaniment — below median)")
axes[1].set_ylim(35, 100)

plt.suptitle(f"Pitch Splitting Result — {entry['track_name']}", fontsize=13)
plt.tight_layout()
plt.show()

### 5d. Separate using the split notes (conditioned model)

In [ ]:
def separate_with_notes(model, mix, notes_roll, device):
    """Separate using conditioned model with external piano roll."""
    ref = mix.mean(0)
    ref_mean, ref_std = ref.mean(), ref.std()
    normalized = (mix - ref_mean) / ref_std

    model_input = torch.cat((normalized, notes_roll), dim=0)

    with torch.no_grad():
        sources = apply_model(model, model_input[None], progress=False, device=device)[0]

    sources = sources * ref_std + ref_mean
    return sources[0].cpu(), sources[1].cpu()


def separate_unconditioned(model, mix, device):
    """Separate using unconditioned model."""
    ref = mix.mean(0)
    ref_mean, ref_std = ref.mean(), ref.std()
    normalized = (mix - ref_mean) / ref_std

    with torch.no_grad():
        sources = apply_model(model, normalized[None], progress=False, device=device)[0]

    sources = sources * ref_std + ref_mean
    return sources[0].cpu(), sources[1].cpu()


print("Separating with Basic Pitch + conditioned model...")
g1_bp, g2_bp = separate_with_notes(model_cond, mix, piano_roll, device)

print("Separating with unconditioned model (baseline)...")
g1_uncond, g2_uncond = separate_unconditioned(model_uncond, mix, device)

print("Done.")

## 6. Compare metrics

In [ ]:
def compute_bss_metrics(g1_ref, g2_ref, g1_est, g2_est, sr):
    min_len = min(g1_ref.shape[-1], g1_est.shape[-1])
    refs = np.stack([
        g1_ref[:, :min_len].T.numpy(),
        g2_ref[:, :min_len].T.numpy(),
    ], axis=0)
    ests = np.stack([
        g1_est[:, :min_len].T.numpy(),
        g2_est[:, :min_len].T.numpy(),
    ], axis=0)
    sdr, sir, isr, sar, _ = windowed_bss_eval(
        refs, ests,
        window=sr, hop=sr,
        compute_permutation=True,
    )
    return {
        "Guitar 1": {"SDR": float(np.nanmedian(sdr[0])), "SIR": float(np.nanmedian(sir[0])), "SAR": float(np.nanmedian(sar[0]))},
        "Guitar 2": {"SDR": float(np.nanmedian(sdr[1])), "SIR": float(np.nanmedian(sir[1])), "SAR": float(np.nanmedian(sar[1]))},
    }


metrics_bp     = compute_bss_metrics(g1_ref, g2_ref, g1_bp,     g2_bp,     sr)
metrics_uncond = compute_bss_metrics(g1_ref, g2_ref, g1_uncond, g2_uncond, sr)

print(f"{'':20s} {'BP+Conditioned':>16s}  {'Unconditioned':>14s}")
print("-" * 54)
for source in ["Guitar 1", "Guitar 2"]:
    for metric in ["SDR", "SIR", "SAR"]:
        v_bp = metrics_bp[source][metric]
        v_u  = metrics_uncond[source][metric]
        delta = v_bp - v_u
        label = f"{source} {metric}"
        arrow = "▲" if delta > 0 else "▼" if delta < 0 else "="
        print(f"{label:20s} {v_bp:+10.2f} dB   {v_u:+10.2f} dB   {arrow} {abs(delta):.2f}")

## 7. Audio comparison

In [ ]:
min_len = min(mix.shape[-1], g1_bp.shape[-1], g1_uncond.shape[-1])

print("=== Mix ===")
ipd.display(ipd.Audio(mix[:, :min_len].numpy(), rate=sr))

for guitar, ref, est_bp, est_u in [
    ("Guitar 1", g1_ref, g1_bp, g1_uncond),
    ("Guitar 2", g2_ref, g2_bp, g2_uncond),
]:
    print(f"\n=== {guitar} — Reference ===")
    ipd.display(ipd.Audio(ref[:, :min_len].numpy(), rate=sr))
    print(f"=== {guitar} — Basic Pitch + Conditioned ===")
    ipd.display(ipd.Audio(est_bp[:, :min_len].numpy(), rate=sr))
    print(f"=== {guitar} — Unconditioned (baseline) ===")
    ipd.display(ipd.Audio(est_u[:, :min_len].numpy(), rate=sr))

## 8. Spectrogram comparison

In [ ]:
def plot_spectrogram(wav, sr, ax, title="", vmin=-60, vmax=0):
    mono = wav.mean(dim=0)[:min_len]
    spec = torch.stft(mono, n_fft=2048, hop_length=512,
                      window=torch.hann_window(2048), return_complex=True)
    mag_db = 20 * torch.log10(spec.abs().clamp(min=1e-8))
    ax.imshow(mag_db.numpy(), aspect="auto", origin="lower",
              extent=[0, min_len/sr, 0, sr/2], cmap="magma", vmin=vmin, vmax=vmax)
    ax.set_ylim(0, 8000)
    ax.set_title(title, fontsize=9)


fig, axes = plt.subplots(2, 3, figsize=(18, 8), sharex=True, sharey=True)

plot_spectrogram(g1_ref,    sr, axes[0, 0], "G1 — Reference")
plot_spectrogram(g1_bp,     sr, axes[0, 1], "G1 — BP + Conditioned")
plot_spectrogram(g1_uncond, sr, axes[0, 2], "G1 — Unconditioned")
plot_spectrogram(g2_ref,    sr, axes[1, 0], "G2 — Reference")
plot_spectrogram(g2_bp,     sr, axes[1, 1], "G2 — BP + Conditioned")
plot_spectrogram(g2_uncond, sr, axes[1, 2], "G2 — Unconditioned")

for ax in axes[-1]:
    ax.set_xlabel("Time (s)")
for ax in axes[:, 0]:
    ax.set_ylabel("Freq (Hz)")

plt.suptitle(f"Improvement 1 — {entry['track_name']}", fontsize=14)
plt.tight_layout()
plt.show()

## 9. Batch evaluation over all test tracks

In [ ]:
all_bp = []
all_uncond = []

for i, entry_i in enumerate(test_entries):
    print(f"[{i+1}/{len(test_entries)}] {entry_i['track_name']}", end=" ")
    mix_i, sr_i = load_audio(entry_i["mix"])
    g1r_i, _ = load_audio(entry_i["sources"]["guitar1"])
    g2r_i, _ = load_audio(entry_i["sources"]["guitar2"])

    try:
        ne_i = transcribe_mix(entry_i["mix"])
        pr_i = median_split_with_hysteresis(ne_i, mix_i.shape[-1], sr=sr_i)
        g1c, g2c = separate_with_notes(model_cond, mix_i, pr_i, device)
        all_bp.append(compute_bss_metrics(g1r_i, g2r_i, g1c, g2c, sr_i))
    except Exception as exc:
        print(f"  BP FAILED: {exc}")
        all_bp.append(None)

    try:
        g1u, g2u = separate_unconditioned(model_uncond, mix_i, device)
        all_uncond.append(compute_bss_metrics(g1r_i, g2r_i, g1u, g2u, sr_i))
    except Exception as exc:
        print(f"  Uncond FAILED: {exc}")
        all_uncond.append(None)

    print("ok")

print("\n=== Aggregate (median over test tracks) ===")
print(f"{'':20s} {'BP+Conditioned':>16s}  {'Unconditioned':>14s}  {'Delta':>8s}")
print("-" * 65)
for source in ["Guitar 1", "Guitar 2"]:
    for metric in ["SDR", "SIR", "SAR"]:
        vals_bp = [m[source][metric] for m in all_bp if m is not None]
        vals_u  = [m[source][metric] for m in all_uncond if m is not None]
        med_bp = np.nanmedian(vals_bp) if vals_bp else float("nan")
        med_u  = np.nanmedian(vals_u) if vals_u else float("nan")
        delta  = med_bp - med_u
        label = f"{source} {metric}"
        print(f"{label:20s} {med_bp:+10.2f} dB   {med_u:+10.2f} dB   {delta:+.2f}")